In [ ]:
import asyncio
import websockets
import json

BENIM_ID = "2300005396"
GIRILECEK_DERS = "Software Engineering" 
SUREKLI_BAGLI_KALSIN = True 
KOPYA_CEKIYOR_MU = False

async def run_student():
    uri = "ws://localhost:8765"
    
    # 🔄 OTO-RECONNECT DÖNGÜSÜ: Bağlantı kopsa bile pes etmez, sürekli dener!
    while True: 
        try:
            async with websockets.connect(uri) as websocket:
                print(f"\n🟢 [ÖĞRENCİ {BENIM_ID}] Sunucuya bağlanıldı!")

                # Sunucuya kimliğimizi bildiriyoruz (Otomatik Giriş İsteği)
                await websocket.send(json.dumps({
                    "action": "request_start_exam",
                    "student_id": BENIM_ID,
                    "exam_id": GIRILECEK_DERS
                }))
                
                response = json.loads(await websocket.recv())
                
                if response.get("status") == "error":
                    print(f"❌ BAĞLANTI REDDEDİLDİ: {response.get('message')}")
                    return # Yasaklıysak tamamen dur
                
                if response.get("status") == "success":
                    token = response.get("session_token")
                    
                    if response.get("warning"):
                        print(f"{response.get('warning')}")

                    # ⏱️ SÜRE SENKRONİZASYONU BURADA YAPILIYOR
                    if response.get("reconnected") == True:
                        # Sunucunun veritabanından getirdiği son süreyi alıp kendi sayacımızı eşitliyoruz!
                        time_left = response.get("time_left_seconds")
                        print(f"<- 🔄 Bağlantı kurtarıldı! Sunucudaki süreye eşitlendi: {time_left} saniye\n")
                    else:
                        duration_mins = response.get("total_duration_minutes", 40)
                        time_left = duration_mins * 60
                        print(f"<- Sınav başladı! Alınan Süre: {duration_mins} dakika ({time_left} saniye)\n")
                    
                    dongu_limiti = 999999 if SUREKLI_BAGLI_KALSIN else 3
                    
                    for i in range(dongu_limiti):
                        await asyncio.sleep(2)
                        
                        if not KOPYA_CEKIYOR_MU:
                            time_left -= 2  
                            durum_mesaji = f"\r-> Güncelleme: Kalan süre {time_left}sn    "
                        else:
                            durum_mesaji = f"\r🚨 YASAKLI İŞLEM TESPİTİ! Yerel süre donduruldu: {time_left}sn    "

                        if time_left <= 0:
                            print("\nSüre bitti!")
                            return

                        await websocket.send(json.dumps({
                            "action": "status_update",
                            "student_id": BENIM_ID,
                            "session_token": token,
                            "timing": {"time_remaining_seconds": time_left},
                            "security": {"violation_alert": KOPYA_CEKIYOR_MU} 
                        }))
                        print(durum_mesaji, end="", flush=True)
                    
                    print(f"\n🔴 [ÖĞRENCİ {BENIM_ID}] Çıkış yapıldı.")
                    break # Normal bir şekilde bittiyse döngüden çık
                    
        except (websockets.exceptions.ConnectionClosedError, ConnectionRefusedError):
            # SUNUCU ÇÖKERSE BURAYA DÜŞER!
            print(f"\n🔌 Sunucu kapandı veya bağlantı koptu! 3 saniye sonra otomatik tekrar denenecek...")
            await asyncio.sleep(3) # Bekle ve döngünün başına dönüp sessizce tekrar bağlan!
        except Exception as e:
            print(f"❌ Beklenmedik Hata: {e}")
            await asyncio.sleep(3)

await run_student()


🟢 [ÖĞRENCİ 2300005396] Sunucuya bağlanıldı!
⚠️ DİKKAT: Eski sürüm (şifresiz) bir istemci kullanıyorsunuz!
<- Sınav başladı! Alınan Süre: 40 dakika (2400 saniye)

-> Güncelleme: Kalan süre 2376sn    ❌ Beklenmedik Hata: received 1001 (going away); then sent 1001 (going away)

🔌 Sunucu kapandı veya bağlantı koptu! 3 saniye sonra otomatik tekrar denenecek...

🔌 Sunucu kapandı veya bağlantı koptu! 3 saniye sonra otomatik tekrar denenecek...

🟢 [ÖĞRENCİ 2300005396] Sunucuya bağlanıldı!
⚠️ DİKKAT: Eski sürüm (şifresiz) bir istemci kullanıyorsunuz!
<- Sınav başladı! Alınan Süre: 40 dakika (2400 saniye)

-> Güncelleme: Kalan süre 2302sn    
🔌 Sunucu kapandı veya bağlantı koptu! 3 saniye sonra otomatik tekrar denenecek...

🟢 [ÖĞRENCİ 2300005396] Sunucuya bağlanıldı!
<- 🔄 Bağlantı kurtarıldı! Sunucudaki süreye eşitlendi: 788 saniye

-> Güncelleme: Kalan süre 784sn    ❌ Beklenmedik Hata: received 1001 (going away); then sent 1001 (going away)

🔌 Sunucu kapandı veya bağlantı koptu! 3 saniye sonra 